In [190]:
import numpy as np


In [191]:
lake=(0,0)
fire=[(4,4)]
smoke=[(1,2),(3,2)]
boulders=[(3,4),(2,4)]


In [192]:
def terminal(state):
    if tuple(state[1:3]) in fire and state[0]:
        return True
    elif tuple(state[1:3]) in boulders:
        return True 
    else:
        return False
    
def reward(state):
    pos = tuple(state[1:3])
    if pos in fire and state[0]:
        return 100
    elif pos in smoke:
        return -10
    elif pos in boulders:
        return -100
    else:
        return -1

def movement(state, dirc):
    if dirc=='N':
        if tuple(state[1:3])==(0,1) and not state[0]:
            return (1, 0,0)
        return state if state[2] == 0 else (state[0], state[1], state[2]-1)
    
    elif dirc=='S':
        return state if state[2] == 4 else (state[0], state[1], state[2]+1)
    
    elif dirc=='E':
        return state if state[1] == 4 else (state[0], state[1]+1, state[2])
    
    elif dirc=='W':
        if tuple(state[1:3])==(1,0) and not state[0]:
            return (1, 0,0)
        return state if state[1] == 0 else (state[0], state[1]-1, state[2])
    
    else:
        return state
    
def possible_states(state,dirc):
    if dirc == 'N':
        return [movement(state, 'N'), movement(state, 'E'), movement(state, 'W'),movement(state, '#')]
    elif dirc == 'S':
        return [movement(state, 'S'), movement(state, 'E'), movement(state, 'W'),movement(state, '#')]
    elif dirc == 'E':
        return [movement(state, 'E'), movement(state, 'N'), movement(state, 'S'),movement(state, '#')]
    elif dirc == 'W':
        return [movement(state, 'W'), movement(state, 'N'), movement(state, 'S'),movement(state, '#')]
    

In [193]:
LOWER_BOUND=-1000
def value_func( state, gamma):
    value=LOWER_BOUND
    if tuple(state[1:3]) in smoke:
        for dirc in ['N','S','E','W']:
            temp_val=0
            p_states = possible_states(state, dirc)
            temp_val += 0.4 * (reward(p_states[0]) + gamma * gridWorld[p_states[0][0], p_states[0][1], p_states[0][2]])
            temp_val += 0.4 * (reward(p_states[3]) + gamma * gridWorld[p_states[3][0], p_states[3][1], p_states[3][2]])
            for i in range(1,3):
                p_state = p_states[i]
                temp_val += 0.1 * (reward(p_state) + gamma * gridWorld[p_state[0], p_state[1], p_state[2]])
            value = max(value, temp_val)
                    
    else:
        for dirc in ['N','S','E','W']:
            temp_val=0
            p_states = possible_states(state, dirc)
            temp_val += 0.7 * (reward(p_states[0]) + gamma * gridWorld[p_states[0][0], p_states[0][1], p_states[0][2]])
            for i in range(1,4):
                p_state = p_states[i]
                temp_val += 0.1 * (reward(p_state) + gamma * gridWorld[p_state[0], p_state[1], p_state[2]])
            value = max(value, temp_val)
        
    return value


In [194]:
def policy_iteration(gamma, theta):
    gridWorld=np.zeros((2,5,5), dtype=float)
    for _ in range(1000):
        delta = 0
        for i in range(1,-1,-1):
            for j in range(4,-1,-1):
                for k in range(4,-1,-1):
                    state = [i,j,k]
                    if not terminal(state):
                        v = gridWorld[i,j,k]
                        gridWorld[i,j,k] = value_func(state, gamma)
                        delta = max(delta, abs(v - gridWorld[i,j,k]))
    
    return gridWorld
                    


In [195]:
gridWorld=policy_iteration(0.9, 0.01)
gridWorld


array([[[  3.0663579 ,   7.84008801,   0.47708109,  -2.79060408,
          -4.60049652],
        [  7.84008801,   2.76569617,  -7.94628673,  -5.94553284,
          -5.64256455],
        [  2.33849806,  -0.79475559,  -6.19101256,  -7.95224851,
           0.        ],
        [ -1.09541724,  -3.05321122, -10.92848181, -17.02397795,
           0.        ],
        [ -3.2447236 ,  -4.40607337,  -6.26464182,  -6.49820171,
          -6.51589249]],

       [[ 15.54343136,  16.66968791,   9.2801412 ,  12.82440421,
           7.92506039],
        [ 23.60239502,  25.49045308,  14.42856384,  19.13063041,
           5.56779236],
        [ 31.88783663,  36.78907913,  30.68654554,  28.66438559,
           0.        ],
        [ 39.99571643,  47.21966633,  47.48923937,  54.62710898,
           0.        ],
        [ 48.24932903,  59.50591897,  72.67590883,  90.97562651,
           0.        ]]])